# 🚀 Hệ thống Nhận diện Rác 2 Giai đoạn (SAHI + Tiled Training)

Notebook này thực thi **TOÀN BỘ** quy trình nâng cao từ đầu đến cuối trên Kaggle.

### 📐 Sự khác biệt so với Pipeline thường:
1. **Tiling:** Cắt bộ dữ liệu ảnh 4K gốc thành các ảnh lưới 640x640.
2. **Stage 1 (Phát hiện):** YOLO26s được huấn luyện trực tiếp trên tập dữ liệu cắt lưới.
3. **Stage 2 (Phân loại):** EfficientNet-B2.
4. **Inference (SAHI):** Cắt ảnh lúc test để chạy YOLO, sau đó gộp Bounding Box lại (NMS) và đưa vào Stage 2 phân loại.


In [1]:
# ============================================================
# 1. Tải Mã nguồn & Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q sahi ultralytics timm


Cloning into 'waste-detection2-Stage'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 191 (delta 36), reused 43 (delta 16), pack-reused 126 (from 1)
Receiving objects: 100% (191/191), 101.30 MiB | 35.14 MiB/s, done.
Resolving deltas: 100% (74/74), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, bu

In [2]:
# ============================================================
# 2. Nhập Dữ liệu Ngoại lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Đã tải: {os.path.basename(task['src'])}")
    else:
        print(f"Bỏ qua: {task['src']} (Không tìm thấy trên Kaggle Dataset)")

Đã tải: dataset-resized
Đã tải: data
Đã tải: RealWaste


In [3]:
# ============================================================
# 3. Chạy Toàn bộ Tiền xử lý Dữ liệu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("\n--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\n--- 3.2 Chuẩn bị dữ liệu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("\n--- 3.3 Tiling dữ liệu Binary cho Stage 1 (Tối ưu cho SAHI) ---")
!python src/data_prep/tiling_binary.py

print("\n✅ Hoàn tất chuẩn bị 100% dữ liệu!")


/kaggle/working/waste-detection2-Stage

--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1
 

In [4]:
# ============================================================
# 4. Huấn luyện Stage 1 (YOLO26s) trên dữ liệu Tiled
# ============================================================
from ultralytics import YOLO
import time
from pathlib import Path

DATASET_YAML = "/kaggle/working/waste-detection2-Stage/data/processed_binary_tiled/dataset.yaml"
OUTPUT_DIR   = "/kaggle/working/waste-detection2-Stage/results/yolo26s_tiled_runs"

print("🚀 Bắt đầu train Stage 1 trên dữ liệu Tiled (YOLO26s – 100 epochs)")
model_s = YOLO("yolo26s.pt")
t0 = time.time()
model_s.train(
    data=DATASET_YAML, imgsz=640, epochs=100, batch=32, patience=20,
    optimizer="auto", lr0=0.01, cos_lr=True, augment=True, workers=4,
    project=OUTPUT_DIR, name="stage1_tiled", exist_ok=True, save=True, save_period=-1,
    device=[0, 1]
)
print(f"\n✅ Train Stage 1 hoàn tất sau {(time.time()-t0)/60:.1f} phút")

!cp results/yolo26s_tiled_runs/stage1_tiled/weights/best.pt stage1_tiled_best.pt


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Bắt đầu train Stage 1 trên dữ liệu Tiled (YOLO26s – 100 epochs)
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/waste-detection2-Stage/data/processed_binary_tiled/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dyna

In [22]:
# Cập nhật code mới từ GitHub (đã sửa lỗi DataParallel)
%cd /kaggle/working/waste-detection2-Stage
!git pull origin main

/kaggle/working/waste-detection2-Stage
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 452 bytes | 226.00 KiB/s, done.
From https://github.com/Shiba-dotcom/waste-detection2-Stage
 * branch            main       -> FETCH_HEAD
   ed7c8f8..8c87aae  main       -> origin/main
Updating ed7c8f8..8c87aae
Fast-forward
 src/evaluate_2stage_sahi.py | 1 +
 1 file changed, 1 insertion(+)


In [18]:
# ============================================================
# 5. Huấn luyện Stage 2 (EfficientNet-B2 – 6 Lớp Classifier)
# Dropout=0.5 | WeightDecay=1e-4 | Mixup(alpha=0.3) | TTA
# ============================================================
!python src/data_prep/generate_background.py
!python src/train_stage2_classifier.py

  GENERATE BACKGROUND CLASS - Stage 2
  Source (Binary) : /kaggle/working/waste-detection2-Stage/data/processed_binary
  Output (Merged) : /kaggle/working/waste-detection2-Stage/data/classification_merged

[XỬ LÝ] Split: TRAIN
Generating train: 100%|█████████████████████████████████████████| 2500/2500 [03:32<00:00, 11.77it/s]
  -> Đã tạo 2500/2500 ảnh Background cho train.

[XỬ LÝ] Split: VAL
Generating val: 100%|█████████████████████████████████████████████| 400/400 [00:26<00:00, 15.01it/s]
  -> Đã tạo 400/400 ảnh Background cho val.

[XỬ LÝ] Split: TEST
Generating test: 100%|████████████████████████████████████████████| 400/400 [00:25<00:00, 15.79it/s]
  -> Đã tạo 400/400 ảnh Background cho test.

  [HOÀN TẤT] Sinh dữ liệu Background thành công!
  Hãy cập nhật train_stage2_classifier.py đổi NUM_CLASSES = 6
  Và CLASS_NAMES = ['Background', 'Glass', 'Metal', 'Other', 'Paper', 'Plastic']
[INFO] PyTorch : 2.10.0+cu128
[INFO] timm    : 1.0.26
[INFO] CUDA    : True
[INFO] GPU     : Tesla 

In [23]:
# ============================================================
# 6. Đánh giá Toàn Trình bằng thuật toán SAHI
# ============================================================
import subprocess, sys, shutil
from pathlib import Path

# Bước 6a: Lọc ra 48 ảnh test chuẩn (có cả nhãn binary và multi-class)
img_binary = Path("data/processed_binary/images/test")
lbl_multi  = Path("data/processed/labels/test")

tmp_img = Path("/kaggle/working/tmp_eval_imgs_strict")
tmp_lbl = Path("/kaggle/working/tmp_eval_lbls_strict")

for d in [tmp_img, tmp_lbl]:
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)

valid_pairs = []
for img_path in sorted(img_binary.rglob("*")):
    if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
    rel_path = img_path.relative_to(img_binary)
    multi_lbl_path = lbl_multi / rel_path.with_suffix('.txt')
    if multi_lbl_path.exists():
        valid_pairs.append((img_path, multi_lbl_path))

for img, lbl in valid_pairs:
    flat_name = "__".join(img.relative_to(img_binary).parts)
    shutil.copy2(img, tmp_img / flat_name)
    shutil.copy2(lbl, tmp_lbl / Path(flat_name).with_suffix('.txt').name)

print(f"[INFO] Đã chuẩn bị {len(valid_pairs)} ảnh test.")

# Bước 6b: Chạy đánh giá SAHI
print("[INFO] Đang chạy SAHI inference (slice=512, overlap=0.2)...")
subprocess.run([
    sys.executable, "src/evaluate_2stage_sahi.py",
    "--detector",   "stage1_tiled_best.pt",
    "--classifier", "models/stage2_best.pth",
    "--data-dir",   str(tmp_img),
    "--label-dir",  str(tmp_lbl),
    "--conf",       "0.35",
    "--slice-size", "512",
    "--overlap",    "0.2"
])
print("\n✅ ĐÁNH GIÁ HOÀN TẤT!")


[INFO] Đã chuẩn bị 50 ảnh test.
[INFO] Đang chạy SAHI inference (slice=512, overlap=0.2)...
[INFO] Khởi tạo SAHI AutoDetectionModel...
[INFO] Khởi tạo EfficientNet-B2 (Stage 2)...
[INFO] num_classes = 6, classes = ['Background', 'Glass', 'Metal', 'Other', 'Paper', 'Plastic']
[INFO] Bắt đầu SAHI Inference trên 24 ảnh...


100%|██████████| 24/24 [00:34<00:00,  1.43s/it]


Performing prediction on 6 slices.
Performing prediction on 50 slices.
Performing prediction on 80 slices.
Performing prediction on 50 slices.
Performing prediction on 48 slices.
Performing prediction on 20 slices.
Performing prediction on 48 slices.
Performing prediction on 50 slices.
Performing prediction on 48 slices.
Performing prediction on 20 slices.
Performing prediction on 48 slices.
Performing prediction on 50 slices.
Performing prediction on 60 slices.
Performing prediction on 50 slices.
Performing prediction on 48 slices.
Performing prediction on 50 slices.
Performing prediction on 80 slices.
Performing prediction on 48 slices.
Performing prediction on 50 slices.
Performing prediction on 80 slices.
Performing prediction on 12 slices.
Performing prediction on 48 slices.
Performing prediction on 50 slices.
Performing prediction on 80 slices.

  KẾT QUẢ ĐÁNH GIÁ (SAHI + EFFICIENTNET)
Tổng GT boxes   : 130
Tổng Pred boxes : 128
IoU matches     : 50 (38.5%)
----------------------

In [24]:
# ============================================================
# 7. Zip tất cả kết quả để tải về
# ============================================================
import shutil, os

shutil.make_archive("/kaggle/working/final_results", "zip", "/kaggle/working/waste-detection2-Stage/results")
print("\n📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results.zip")

if os.path.exists("/kaggle/working/waste-detection2-Stage/models"):
    shutil.make_archive("/kaggle/working/final_models", "zip", "/kaggle/working/waste-detection2-Stage/models")
    print("📦 Đã nén trọng số mô hình: /kaggle/working/final_models.zip")



📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results.zip
📦 Đã nén trọng số mô hình: /kaggle/working/final_models.zip
